<a href="https://colab.research.google.com/github/HebaRouk/-Build-a-ML-Workflow-For-Scones-Unlimited-On-Amazon-SageMaker/blob/main/lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving sol_2b (2).py to sol_2b (2).py
Saving AI7102_Lab2 (3).pdf to AI7102_Lab2 (3).pdf
Saving AI7102_Students_Lab2 (2).zip to AI7102_Students_Lab2 (2).zip
['sol_2b (2).py', 'AI7102_Lab2 (3).pdf', 'AI7102_Students_Lab2 (2).zip']


In [ ]:
%cd /content
!rm -rf lab
!unzip -q -o "/content/AI7102_Students_Lab2 (2).zip" -d lab

import glob, os
hits = glob.glob('/content/lab/**/lab.py', recursive=True)
print(hits)
if hits:
    %cd {os.path.dirname(hits[0])}
    !ls
else:
    !ls -R /content/lab | head -40

/content
['/content/lab/ai7102-student/lab.py']
/content/lab/ai7102-student
AGENTS.md     CLAUDE.md  lab.py		 README.md  uv.lock
AI_POLICY.md  lab2	 pyproject.toml  shared


In [ ]:
!pip install -q uv
!uv sync --locked
!uv run --locked python lab.py check 2

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 43 packages in 0.80ms
Prepared 39 packages in 6.18s
Installed 39 packages in 344ms
 + cffi==2.1.1
 + cloudpickle==3.1.2
 + contourpy==1.4.0
 + cryptography==46.0.4
 + cycler==0.12.1
 + et-xmlfile==2.0.0
 + filelock==4.0.8
 + fonttools==4.66.1
 + fsspec==2026.9.0
 + iniconfig==2.3.0
 + jinja2==3.1.6
 + joblib==1.6.0
 + kiwisolver==1.5.1
 + markupsafe==3.0.3
 + matplotlib==3.10.8
 + mpmath==1.3.0
 + networkx==3.7
 + numpy==2.3.5
 + openpyxl==3.1.5
 + packaging==26.3
 + pandas==2.2.3
 + pillow==12.3.0
 + pluggy==1.6.0
 + pycparser==3.0
 + pygments==2.21.0
 + pyparsing==3.3.3
 + pytest==9.0.2
 + python-dateutil==2.9.0.post0
 + pytz==2026.4
 + scikit-learn==1.8.0
 + scipy==1.18.1
 + setuptools==84.0.0
 + six==1.17.0
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + torch==2.10.0+cpu
 + torchvision==0.25.0+cpu
 + typing-extensions==4.16.0
 + tzdata==2026.4
ERROR: Unexpected PYTHONPATH. Clear it

In [ ]:
import os
os.environ.pop('PYTHONPATH', None)
!uv run --locked python lab.py check 2

{
  "release": "-lab-2",
  "available_labs": [
    2
  ],
  "audience": "student",
  "checked_files": 30,
  "authority": "local repository key (feedback only)"
}


In [ ]:
!uv run --locked python lab.py data raisin --preview
!uv run --locked python lab.py data banknote --preview

Lab 2 dataset: UCI raisin
Label -1: Besni. Label +1: Kecimen.
Examples: train=576, validation=144, test=180
First five training rows, in original measurement units:
 sample_id      Area  Eccentricity  label
       254 74109.000         0.833      1
       449 41995.000         0.584      1
       825 61556.000         0.734     -1
       624 71800.000         0.862     -1
       474 83800.000         0.831     -1
Lab 2 dataset: UCI banknote
Label -1: 0. Label +1: 1.
Examples: train=877, validation=220, test=275
First five training rows, in original measurement units:
 sample_id  variance  skewness  label
       779    -1.819    -9.037      1
       813    -2.133     1.569      1
      1174    -3.205    -0.143      1
      1037    -1.798    -6.769      1
      1220    -2.777    -5.697      1


In [ ]:
%%writefile lab2/student.py
"""Lab 2 functions. Read the separate Lab 2 sheet on the LMS for the tasks."""
from collections.abc import Callable

import numpy as np


def _check_label(y_i) -> int:
    if isinstance(y_i, (bool, np.bool_)) or y_i not in (-1, 1):
        raise ValueError(f'label must be -1 or +1, got {y_i!r}')
    return int(y_i)


def _check_learning_rate(learning_rate: float) -> float:
    lr = float(learning_rate)
    if not np.isfinite(lr) or lr <= 0:
        raise ValueError(f'learning rate must be positive and finite, got {learning_rate!r}')
    return lr


def perceptron_predict(x: np.ndarray, weight: np.ndarray, bias: float) -> np.ndarray:
    """1(a): Return signed labels of shape (N,) for a batch of shape (N, d)."""
    scores = np.asarray(x, dtype=float) @ np.asarray(weight, dtype=float) + bias
    return np.where(scores >= 0, 1, -1).astype(int)


def perceptron_update(
    x_i: np.ndarray,
    y_i: int,
    weight: np.ndarray,
    bias: float,
    learning_rate: float,
) -> tuple[np.ndarray, float, bool]:
    """1(b): Return the weights, bias and update flag for one example."""
    y_i = _check_label(y_i)
    lr = _check_learning_rate(learning_rate)
    x_i = np.asarray(x_i, dtype=float)
    weight = np.array(weight, dtype=float)

    margin = y_i * (x_i @ weight + bias)
    if margin <= 0:
        return weight + lr * y_i * x_i, float(bias + lr * y_i), True
    return weight, float(bias), False


def classification_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """1(c): Return the fraction of labels that differ."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_true.shape != y_pred.shape:
        raise ValueError(f'shape mismatch: {y_true.shape} vs {y_pred.shape}')
    if y_true.size == 0:
        raise ValueError('label arrays must be nonempty')
    return float(np.mean(y_true != y_pred))


def average_mismatch(
    scores: np.ndarray,
    targets: np.ndarray,
    mismatch_fn: Callable[[np.ndarray, np.ndarray], np.ndarray],
) -> float:
    """1(c): Average the values returned by mismatch_fn."""
    values = np.asarray(mismatch_fn(scores, targets), dtype=float)
    if values.size == 0:
        raise ValueError('mismatch_fn returned no values')
    return float(np.mean(values))


def fit_perceptron(
    x: np.ndarray,
    y: np.ndarray,
    learning_rate: float,
    max_epochs: int,
) -> dict:
    """2(a): Train from zero parameters. Return the counts and histories in the sheet."""
    if max_epochs <= 0:
        raise ValueError(f'max_epochs must be positive, got {max_epochs!r}')
    lr = _check_learning_rate(learning_rate)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y)
    if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] == 0:
        raise ValueError(f'x must have shape (N, d) with N, d > 0, got {x.shape}')
    if y.shape != (x.shape[0],):
        raise ValueError(f'y must have shape ({x.shape[0]},), got {y.shape}')
    if y.dtype == bool or not np.all(np.isin(y, (-1, 1))):
        raise ValueError('labels must be -1 or +1')

    weight = np.zeros(x.shape[1])
    bias = 0.0
    mistakes_per_epoch: list[int] = []
    updates_per_epoch: list[int] = []
    weight_history = [weight.copy()]
    bias_history = [bias]
    converged = False

    for _ in range(int(max_epochs)):
        mistakes = 0
        updates = 0
        for x_i, y_i in zip(x, y):
            if perceptron_predict(x_i[None, :], weight, bias)[0] != y_i:
                mistakes += 1
            weight, bias, changed = perceptron_update(x_i, int(y_i), weight, bias, lr)
            updates += int(changed)
        mistakes_per_epoch.append(mistakes)
        updates_per_epoch.append(updates)
        weight_history.append(weight.copy())
        bias_history.append(bias)
        if updates == 0:
            converged = True
            break

    return {
        'weight': weight,
        'bias': float(bias),
        'mistakes_per_epoch': mistakes_per_epoch,
        'updates_per_epoch': updates_per_epoch,
        'weight_history': weight_history,
        'bias_history': bias_history,
        'epochs': len(updates_per_epoch),
        'converged': converged,
    }


def observed_class_proportions(
    feature: np.ndarray,
    labels_zero_one: np.ndarray,
    bin_edges: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """2(b): Return bin midpoints and class-1 fractions. Empty bins give NaN fractions."""
    feature = np.asarray(feature, dtype=float)
    labels = np.asarray(labels_zero_one)
    edges = np.asarray(bin_edges, dtype=float)

    if feature.ndim != 1 or labels.ndim != 1 or feature.shape != labels.shape:
        raise ValueError('feature and labels must be 1D arrays of the same length')
    if labels.size and (labels.dtype == bool or not np.all(np.isin(labels, (0, 1)))):
        raise ValueError('labels must be 0 or 1')
    if not np.all(np.isfinite(feature)):
        raise ValueError('feature values must be finite')
    if edges.ndim != 1 or edges.size < 2:
        raise ValueError('bin_edges must be a 1D array with at least two edges')
    if not np.all(np.isfinite(edges)):
        raise ValueError('bin edges must be finite')
    if not np.all(np.diff(edges) > 0):
        raise ValueError('bin edges must be strictly increasing')
    if feature.size and (feature.min() < edges[0] or feature.max() > edges[-1]):
        raise ValueError('feature values must lie within the edge range')

    midpoints = (edges[:-1] + edges[1:]) / 2.0
    totals, _ = np.histogram(feature, bins=edges)
    ones, _ = np.histogram(feature[labels == 1], bins=edges)

    fractions = np.full(totals.shape, np.nan, dtype=float)
    nonempty = totals > 0
    fractions[nonempty] = ones[nonempty] / totals[nonempty]
    return midpoints.astype(float), fractions

Overwriting lab2/student.py


In [ ]:
!uv run --locked python lab.py test 2

............................................                             [100%]
44 passed in 0.17s


In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'
!uv run --locked python lab.py run 2

/content/lab/ai7102-student/lab2/results/core/20261112T195321-cb1274a3/index.html
View: uv run --locked python lab.py view 2


In [ ]:
RUN = '20261112T195321-cb1274a3'
!cat lab2/results/core/{RUN}/lab2_summary.json
!echo "================ QUESTION 4 ================"
!cat lab2/results/core/{RUN}/question4/lab2_summary.json

!zip -qr /content/lab2_results.zip lab2/results/core/{RUN} lab2/student.py
from google.colab import files
files.download('/content/lab2_results.zip')

{
  "data": {
    "dataset": "UCI raisin",
    "feature_names": [
      "Area",
      "Eccentricity"
    ],
    "label_minus_1": "Besni",
    "label_plus_1": "Kecimen",
    "training_examples": 576,
    "validation_examples": 144
  },
  "models": {
    "two_feature": {
      "validation_error": 0.2847222222222222,
      "epochs": 35,
      "final_updates": 131,
      "stopping_reason": "epoch limit"
    },
    "all_features": {
      "validation_error": 0.2708333333333333,
      "epochs": 35,
      "final_updates": 107,
      "stopping_reason": "epoch limit"
    }
  }
}
================ QUESTION 4 ================
{
  "data": {
    "dataset": "UCI banknote",
    "feature_names": [
      "variance",
      "skewness"
    ],
    "label_minus_1": "0",
    "label_plus_1": "1",
    "training_examples": 877,
    "validation_examples": 220
  },
  "models": {
    "two_feature": {
      "validation_error": 0.2727272727272727,
      "epochs": 35,
      "final_updates": 140,
      "stopping_reason

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving lab2_results.zip to lab2_results.zip
Saving answers.PDF to answers.PDF
Saving AI7102_Students_Lab2 (2).zip to AI7102_Students_Lab2 (2).zip
['lab2_results.zip', 'answers.PDF', 'AI7102_Students_Lab2 (2).zip']


In [3]:
%cd /content
!rm -rf lab
!unzip -q -o "/content/AI7102_Students_Lab2 (2).zip" -d lab
import glob, os
%cd {os.path.dirname(glob.glob('/content/lab/**/lab.py', recursive=True)[0])}
!ls

/content
/content/lab/ai7102-student
AGENTS.md     CLAUDE.md  lab.py		 README.md  uv.lock
AI_POLICY.md  lab2	 pyproject.toml  shared


In [4]:
os.environ.pop('PYTHONPATH', None)
os.environ['MPLBACKEND'] = 'Agg'
!pip install -q uv
!uv sync --locked
!uv run --locked python lab.py check 2

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 43 packages in 0.91ms
Prepared 39 packages in 4.60s
Installed 39 packages in 454ms
 + cffi==2.1.1
 + cloudpickle==3.1.2
 + contourpy==1.4.0
 + cryptography==46.0.4
 + cycler==0.12.1
 + et-xmlfile==2.0.0
 + filelock==4.0.8
 + fonttools==4.66.1
 + fsspec==2026.9.0
 + iniconfig==2.3.0
 + jinja2==3.1.6
 + joblib==1.6.0
 + kiwisolver==1.5.1
 + markupsafe==3.0.3
 + matplotlib==3.10.8
 + mpmath==1.3.0
 + networkx==3.7
 + numpy==2.3.5
 + openpyxl==3.1.5
 + packaging==26.3
 + pandas==2.2.3
 + pillow==12.3.0
 + pluggy==1.6.0
 + pycparser==3.0
 + pygments==2.21.0
 + pyparsing==3.3.3
 + pytest==9.0.2
 + python-dateutil==2.9.0.post0
 + pytz==2026.4
 + scikit-learn==1.8.0
 + scipy==1.18.1
 + setuptools==84.0.0
 + six==1.17.0
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + torch==2.10.0+cpu
 + torchvision==0.25.0+cpu
 + typing-extensions==4.16.0
 + tzdata==2026.4
{
  "release": "-lab-2",
  "available_

In [5]:
!unzip -o -q /content/lab2_results.zip -d .
!ls lab2/results/core/
!uv run --locked python lab.py test 2

20261112T195321-cb1274a3
............................................                             [100%]
44 passed in 0.18s


In [7]:
!cp /content/answers.PDF lab2/answers.pdf
!ls -l lab2/answers.pdf
!uv run --locked python lab.py package 2 --run-id 20261112T195321-cb1274a3

-rw-r--r-- 1 root root 995548 Nov 13 17:51 lab2/answers.pdf
SHA256: 35eeea7486dc6319a58f98875f135ae7af674954a9ce2c1a15a751ad8969a038
Upload this ZIP and its run ID to the lab submission in the LMS. Keep a copy.
/content/lab/ai7102-student/lab2/submissions/core-20261112T195321-cb1274a3.zip


In [9]:
from google.colab import files
files.download('/content/lab/ai7102-student/lab2/submissions/core-20261112T195321-cb1274a3.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>